<a href="https://colab.research.google.com/github/MariamSheref/dvt-medical-rag/blob/main/AI_Hackathon_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hackthoon

In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 70.2 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [ ]:
!pip install langchain-text-splitters

In [ ]:
import os
import re
import json
import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
from google.colab import files
uploaded = files.upload()

## 1. CONFIG — map each PDF to the source metadata you want cited in answers

In [ ]:
DATA_DIR = "."
SOURCES = [
    {
        "filename": "CDC_About_VTE_DVT.pdf",
        "source_name": "CDC",
        "title": "About Venous Thromboembolism (Blood Clots)",
        "url": "https://www.cdc.gov/blood-clots/about/index.html",
    },
    {
        "filename": "NHS_DVT.pdf",
        "source_name": "NHS",
        "title": "DVT (deep vein thrombosis)",
        "url": "https://www.nhs.uk/conditions/deep-vein-thrombosis-dvt/",
    },
]

# 2. DOCUMENT LOADING — extract raw text page by page


In [ ]:
def load_pdf_text(filepath: str) -> str:
    """Extract all text from a PDF, page by page, and join it."""
    pages_text = []
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            pages_text.append(text)
    return "\n".join(pages_text)


# 3. TEXT CLEANING — remove noise so embeddings aren't polluted by junk


In [ ]:
def clean_text(raw_text: str) -> str:
    """
    Clean extracted PDF text:
    - collapse repeated whitespace/newlines
    - drop page-number artifacts (e.g. "3 of 9")
    - strip stray control characters
    - normalize bullet characters
    """
    text = raw_text

    # Remove "X of Y" page-footer artifacts left over from the PDF layout
    text = re.sub(r"\b\d+\s+of\s+\d+\b", " ", text)

    # Normalize different bullet/dash characters to a plain "-"
    text = re.sub(r"[•◦▪‣]", "-", text)

    # pdfplumber sometimes can't map a bullet glyph to a real character and
    # leaves a raw font code instead, e.g. "(cid:127)" — strip those
    text = re.sub(r"\(cid:\d+\)", "-", text)

    # Collapse multiple spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse 3+ newlines down to a double newline (paragraph break)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Strip leading/trailing whitespace on each line
    text = "\n".join(line.strip() for line in text.split("\n"))

    # Remove any leftover non-printable characters
    text = re.sub(r"[^\x20-\x7E\n]", "", text)

    return text.strip()


# 4. CHUNKING — split into overlapping, semantically coherent chunks


In [ ]:
def chunk_text(text: str, chunk_size: int = 500, chunk_overlap: int = 80):
    """
    RecursiveCharacterTextSplitter tries to split on paragraph breaks first,
    then sentences, then words — so chunks stay semantically coherent
    instead of being cut mid-sentence. Overlap keeps context from being
    lost at chunk boundaries (important for medical Q&A accuracy).
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_text(text)



# 5. PIPELINE — run load -> clean -> chunk for every source, attach metadata


In [ ]:
def build_chunks():
    all_chunks = []
    chunk_id = 0

    for source in SOURCES:
        filepath = os.path.join(DATA_DIR, source["filename"])
        if not os.path.exists(filepath):
            print(f"[WARN] Missing file, skipping: {filepath}")
            continue

        print(f"Loading: {source['filename']}")
        raw_text = load_pdf_text(filepath)

        print(f"Cleaning text ({len(raw_text)} raw chars)...")
        cleaned = clean_text(raw_text)

        print(f"Chunking ({len(cleaned)} clean chars)...")
        chunks = chunk_text(cleaned)

        for chunk in chunks:
            all_chunks.append({
                "id": f"chunk_{chunk_id:04d}",
                "text": chunk,
                "source_name": source["source_name"],   # e.g. "CDC" or "NHS"
                "title": source["title"],
                "url": source["url"],
            })
            chunk_id += 1

        print(f" -> {len(chunks)} chunks from {source['source_name']}\n")

    return all_chunks


if __name__ == "__main__":
    chunks = build_chunks()

    os.makedirs("./output", exist_ok=True)
    out_path = "./output/chunks.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

    print(f"Done. {len(chunks)} total chunks written to {out_path}")
    print("\nSample chunk:")
    print(json.dumps(chunks[0], indent=2, ensure_ascii=False) if chunks else "No chunks produced.")

# VERIFICATION — run this AFTER build_chunks() to sanity-check the extraction



In [ ]:
from collections import Counter

print("=" * 70)
print(f"TOTAL CHUNKS: {len(chunks)}")
print("=" * 70)

# 1) How many chunks came from each source? (should be > 0 for both)
counts = Counter(c["source_name"] for c in chunks)
print("\nChunks per source:")
for source, count in counts.items():
    print(f"  {source}: {count} chunks")

# 2) Show 3 sample chunks (first, middle, last) so you can eyeball quality
print("\n" + "=" * 70)
print("SAMPLE CHUNKS")
print("=" * 70)

sample_indices = [0, len(chunks) // 2, len(chunks) - 1]
for i in sample_indices:
    c = chunks[i]
    print(f"\n--- Chunk {c['id']}  (source: {c['source_name']}) ---")
    print(f"Title: {c['title']}")
    print(f"URL:   {c['url']}")
    print(f"Length: {len(c['text'])} chars")
    print("Text:")
    print(c["text"])
    print("-" * 70)

# 3) Check for leftover extraction artifacts that should have been cleaned
print("\n" + "=" * 70)
print("ARTIFACT CHECK (should all say 'OK — none found')")
print("=" * 70)

import re
checks = {
    "cid: font codes":     r"\(cid:\d+\)",
    "'X of Y' page numbers": r"\b\d+\s+of\s+\d+\b",
    "non-ASCII characters": r"[^\x20-\x7E\n]",
    "triple+ newlines":     r"\n{3,}",
}

all_text = "\n".join(c["text"] for c in chunks)
for label, pattern in checks.items():
    matches = re.findall(pattern, all_text)
    status = "OK — none found" if not matches else f"FOUND {len(matches)} — e.g. {matches[:3]}"
    print(f"  {label}: {status}")

# 4) Check chunk sizes are within a sane range (no empty or giant chunks)
lengths = [len(c["text"]) for c in chunks]
print("\n" + "=" * 70)
print("CHUNK SIZE STATS")
print("=" * 70)
print(f"  Min length: {min(lengths)} chars")
print(f"  Max length: {max(lengths)} chars")
print(f"  Avg length: {sum(lengths) / len(lengths):.0f} chars")

empty_chunks = [c for c in chunks if len(c["text"].strip()) == 0]
print(f"  Empty chunks: {len(empty_chunks)}  (should be 0)")

In [ ]:
!pip uninstall -y pillow
!pip install pillow==11.0.0
!pip install --upgrade sentence-transformers chromadb

In [ ]:
!python embed_and_store.py

In [ ]:
import json
import chromadb
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# 1. CONFIG
# ---------------------------------------------------------------------------

CHUNKS_PATH = "./output/chunks.json"          # output from parse_documents.py
CHROMA_DIR = "./chroma_db"                    # where the vector db is persisted
COLLECTION_NAME = "dvt_medical_sources"

# all-MiniLM-L6-v2: small, fast, free, good enough quality for a demo RAG.
# Runs on CPU, no GPU or API key required.
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"


# ---------------------------------------------------------------------------
# 2. LOAD CHUNKS produced by parse_documents.py
# ---------------------------------------------------------------------------

def load_chunks(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ---------------------------------------------------------------------------
# 3. EMBED + STORE — encode every chunk and add it to a Chroma collection
# ---------------------------------------------------------------------------

def build_vector_store(chunks):
    print(f"Loading embedding model: {EMBEDDING_MODEL_NAME} ...")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    print("Connecting to Chroma (persistent, on disk) ...")
    client = chromadb.PersistentClient(path=CHROMA_DIR)

    # Fresh start each run: drop the collection if it already exists so we
    # don't end up with duplicate chunks after re-running the pipeline.
    existing = [c.name for c in client.list_collections()]
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)

    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},  # cosine similarity for retrieval
    )

    texts = [c["text"] for c in chunks]
    ids = [c["id"] for c in chunks]
    metadatas = [
        {
            "source_name": c["source_name"],
            "title": c["title"],
            "url": c["url"],
        }
        for c in chunks
    ]

    print(f"Embedding {len(texts)} chunks ...")
    embeddings = model.encode(texts, show_progress_bar=True).tolist()

    print("Writing to Chroma collection ...")
    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas,
    )

    print(f"Done. Collection '{COLLECTION_NAME}' now has {collection.count()} chunks.")
    return collection, model


# ---------------------------------------------------------------------------
# 4. QUICK TEST — run a sample retrieval query to confirm it works end-to-end
# ---------------------------------------------------------------------------

def test_query(collection, model, query: str, top_k: int = 3):
    print(f"\nTest query: \"{query}\"")
    print("-" * 70)

    query_embedding = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
    )

    for rank, (doc, meta, dist) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
        start=1,
    ):
        print(f"\n[{rank}] source: {meta['source_name']}  |  distance: {dist:.4f}")
        print(f"    url: {meta['url']}")
        preview = doc[:200].replace("\n", " ")
        print(f"    text: {preview}...")


if __name__ == "__main__":
    chunks = load_chunks(CHUNKS_PATH)
    collection, model = build_vector_store(chunks)

    # Sanity-check retrieval with a couple of realistic DVT questions
    test_query(collection, model, "What are the symptoms of DVT?")
    test_query(collection, model, "How can I prevent a blood clot on a long flight?")
    test_query(collection, model, "What is the link between DVT and pulmonary embolism?")